In [19]:
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

URBAN_API = "https://urban-api.idu.kanootoko.org/api/v1"

#211
#198
#124
#335

id = 198

scenario_response = requests.get(
    f"{URBAN_API}/scenarios/{id}"
)
if scenario_response.status_code != 200:
    raise Exception("Ошибка при получении информации по сценарию")

scenario_data = scenario_response.json()
project_id = scenario_data.get("project", {}).get("project_id")
if project_id is None:
    raise Exception("Project ID is missing in scenario data.")

territory_response = requests.get(
    f"{URBAN_API}/projects/{project_id}/territory"
        )

if territory_response.status_code != 200:
            raise Exception("Ошибка при получении геометрии территории")
        
territory_data = territory_response.json()
territory_geometry = territory_data["geometry"]
territory_feature = {
    'type': 'Feature',
    'geometry': territory_geometry,
    'properties': {}
}
polygon_gdf = gpd.GeoDataFrame.from_features([territory_feature], crs=4326)
polygon_gdf = polygon_gdf.to_crs(32636)


scenario_indicators_response = requests.get(
    f"{URBAN_API}/scenarios/{id}/indicators_values"
)
if scenario_response.status_code != 200:
    raise Exception("Ошибка при получении информации по сценарию")

scenario_indicators = scenario_indicators_response.json()

indicator_attributes = {
    indicator['indicator']['name_full']: indicator['value']
    for indicator in scenario_indicators
}

# Добавляем эти значения как колонки в polygon_gdf
for name, value in indicator_attributes.items():
    polygon_gdf[name] = value
    
polygon_gdf



,geometry,Численность населения,Население,Транспортное обеспечение,Экологическая ситуация,Обеспечение инженерной инфраструктурой,Социальное обеспечение,Средняя этажность,Коэффициент застройки,Коэффициент плотности застройки,...,Потенциал развития среднеэтажной жилой застройки,Потенциал развития многоэтажной жилой застройки,Потенциал развития застройки рекреационной зоны,Потенциал развития застройки сельскохозяйственной зоны,Потенциал развития застройки зоны специального назначения,Потенциал развития застройки промышленной зоны,Потенциал развития застройки транспортной зоны,Срок рекультивации территории,Стоимость рекультивации территории,Плотность населения
0,"POLYGON ((387861.999 6644712.938, 388202.551 6...",204314.0,4.0,5.0,-3.2,5.0,3.5,2.0,0.2,0.05,...,3.0,1.0,4.0,3.0,4.0,4.0,4.0,6555.0,2.284321e+09,89.0


In [20]:
polygon_gdf['Потенциал развития жилой застройки'] = 4
polygon_gdf

,geometry,Численность населения,Население,Транспортное обеспечение,Экологическая ситуация,Обеспечение инженерной инфраструктурой,Социальное обеспечение,Средняя этажность,Коэффициент застройки,Коэффициент плотности застройки,...,Потенциал развития многоэтажной жилой застройки,Потенциал развития застройки рекреационной зоны,Потенциал развития застройки сельскохозяйственной зоны,Потенциал развития застройки зоны специального назначения,Потенциал развития застройки промышленной зоны,Потенциал развития застройки транспортной зоны,Срок рекультивации территории,Стоимость рекультивации территории,Плотность населения,Потенциал развития жилой застройки
0,"POLYGON ((387861.999 6644712.938, 388202.551 6...",204314.0,4.0,5.0,-3.2,5.0,3.5,2.0,0.2,0.05,...,1.0,4.0,3.0,4.0,4.0,4.0,6555.0,2.284321e+09,89.0,4


In [21]:
from typing import Final
import geopandas as gpd

LAND_USE_TO_POTENTIAL_COLUMN: Final[dict[str, str]] = {
    "residential": "Потенциал развития жилой застройки",
    "residential_individual": "Потенциал развития жилой застройки типа ИЖС",
    "residential_lowrise":     "Потенциал развития малоэтажной жилой застройки",
    "residential_midrise":     "Потенциал развития среднеэтажной жилой застройки",
    "residential_multistorey": "Потенциал развития многоэтажной жилой застройки",
    "business":     "Потенциал развития застройки общественно-деловой зоны",
    "recreation":   "Потенциал развития застройки рекреационной зоны",
    "special":      "Потенциал развития застройки зоны специального назначения",
    "industrial":   "Потенциал развития застройки промышленной зоны",
    "agriculture":  "Потенциал развития застройки сельскохозяйственной зоны",
    "transport":    "Потенциал развития застройки транспортной зоны",
}

records = []
for _, row in polygon_gdf.iterrows():
    for ip_type, col_name in LAND_USE_TO_POTENTIAL_COLUMN.items():
        records.append({
            "ip_type":  ip_type,
            "ip_value": row[col_name],
            "geometry": row.geometry
        })

# Создаём новый GeoDataFrame
base_gdf = gpd.GeoDataFrame(records, crs=polygon_gdf.crs)
base_gdf = base_gdf.reset_index(drop=True)
base_gdf



,ip_type,ip_value,geometry
0,residential,4.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
1,residential_individual,2.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
2,residential_lowrise,3.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
3,residential_midrise,3.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
4,residential_multistorey,1.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
5,business,2.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
6,recreation,4.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
7,special,4.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
8,industrial,4.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."
9,agriculture,3.0,"POLYGON ((387861.999 6644712.938, 388202.551 6..."


In [22]:
from urbanomy.methods.investment_potential import LandUseScoreAnalyzer

analyzer = LandUseScoreAnalyzer(weights=None)
score_gdf = analyzer.compute_scores_long(polygon_gdf)
score_gdf

,geometry,ip_type,ip_value
0,"POLYGON ((387861.999 6644712.938, 388202.551 6...",residential,1.5
1,"POLYGON ((387861.999 6644712.938, 388202.551 6...",residential_individual,0.7
2,"POLYGON ((387861.999 6644712.938, 388202.551 6...",residential_lowrise,1.1
3,"POLYGON ((387861.999 6644712.938, 388202.551 6...",residential_midrise,1.2
4,"POLYGON ((387861.999 6644712.938, 388202.551 6...",residential_multistorey,0.4
5,"POLYGON ((387861.999 6644712.938, 388202.551 6...",business,0.8
6,"POLYGON ((387861.999 6644712.938, 388202.551 6...",recreation,1.4
7,"POLYGON ((387861.999 6644712.938, 388202.551 6...",special,1.5
8,"POLYGON ((387861.999 6644712.938, 388202.551 6...",industrial,1.5
9,"POLYGON ((387861.999 6644712.938, 388202.551 6...",agriculture,1.0


In [23]:
import geopandas as gpd
import random
from shapely.geometry import Point
from shapely.ops import voronoi_diagram

def generate_voronoi_zones(ip_long_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:

    base_poly = ip_long_gdf.geometry.unary_union

    ip_map = dict(zip(ip_long_gdf['ip_type'], ip_long_gdf['ip_value']))
    ip_types = list(ip_map.keys())


    minx, miny, maxx, maxy = base_poly.bounds
    random.seed(42)
    seed_points = []
    for _ in ip_types:
        while True:
            x = random.uniform(minx, maxx)
            y = random.uniform(miny, maxy)
            pt = Point(x, y)
            if pt.within(base_poly):
                seed_points.append(pt)
                break

    points_gdf = gpd.GeoDataFrame(
        {'ip_type': ip_types, 'ip_value': [ip_map[t] for t in ip_types]},
        geometry=seed_points,
        crs=ip_long_gdf.crs
    )

    vor = voronoi_diagram(points_gdf.unary_union, envelope=base_poly)
    vor_polys = list(vor.geoms) if hasattr(vor, 'geoms') else list(vor)

    # 6. Для каждой ячейки ищем, к какому seed-поинту она относится
    zones = []
    for cell in vor_polys:
        for _, seed in points_gdf.iterrows():
            if seed.geometry.within(cell):
                zones.append({
                    'geometry': cell,
                    'ip_type':  seed.ip_type,
                    'ip_value': seed.ip_value
                })
                break

    zones_gdf = gpd.GeoDataFrame(zones, crs=ip_long_gdf.crs)

    zones_gdf = gpd.clip(zones_gdf, base_poly)

    return zones_gdf



project_gdf = generate_voronoi_zones(score_gdf) #base_gdf
project_gdf


/var/folders/h8/0wmx2zx90bn60jw8zc1r4qjc0000gn/T/ipykernel_2998/1591258819.py:8: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  base_poly = ip_long_gdf.geometry.unary_union
/var/folders/h8/0wmx2zx90bn60jw8zc1r4qjc0000gn/T/ipykernel_2998/1591258819.py:32: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  vor = voronoi_diagram(points_gdf.unary_union, envelope=base_poly)


,geometry,ip_type,ip_value
4,"POLYGON ((389429.389 6643590.914, 389460.139 6...",residential_midrise,1.2
2,"POLYGON ((388703.113 6643817.106, 388934.446 6...",residential_individual,0.7
6,"POLYGON ((388934.446 6643708.859, 389384.208 6...",industrial,1.5
1,"POLYGON ((388634.582 6644166.028, 388703.113 6...",residential_multistorey,0.4
3,"POLYGON ((388703.113 6643817.106, 388634.582 6...",transport,1.4
0,"POLYGON ((388634.582 6644166.028, 387644.932 6...",recreation,1.4
9,"POLYGON ((389460.139 6643570.686, 390172.418 6...",residential,1.5
8,"POLYGON ((390172.418 6643549.388, 390334.954 6...",special,1.5
5,"POLYGON ((389429.389 6643590.914, 389384.208 6...",business,0.8
10,"POLYGON ((390334.954 6643620.983, 390237.235 6...",agriculture,1.0


In [24]:
project_gdf['area'] = project_gdf.geometry.area
project_gdf

,geometry,ip_type,ip_value,area
4,"POLYGON ((389429.389 6643590.914, 389460.139 6...",residential_midrise,1.2,228946.535860
2,"POLYGON ((388703.113 6643817.106, 388934.446 6...",residential_individual,0.7,555093.622474
6,"POLYGON ((388934.446 6643708.859, 389384.208 6...",industrial,1.5,338108.036466
1,"POLYGON ((388634.582 6644166.028, 388703.113 6...",residential_multistorey,0.4,560271.440910
3,"POLYGON ((388703.113 6643817.106, 388634.582 6...",transport,1.4,850917.184585
0,"POLYGON ((388634.582 6644166.028, 387644.932 6...",recreation,1.4,811927.195847
9,"POLYGON ((389460.139 6643570.686, 390172.418 6...",residential,1.5,441870.693343
8,"POLYGON ((390172.418 6643549.388, 390334.954 6...",special,1.5,409027.820815
5,"POLYGON ((389429.389 6643590.914, 389384.208 6...",business,0.8,817246.640517
10,"POLYGON ((390334.954 6643620.983, 390237.235 6...",agriculture,1.0,591467.024316


In [25]:
benchmarks_demo = {
    "residential": {
        "density": 0.25,
        # "built_area": 110468,
        "land_cost": 1500,
        "cost_build": 45_000,
        "price_sale": 90_000,
        "construction_years": 2,
        "sale_years": 3,
        "opex_rate": 800,
    },
    "residential_individual": {
        "density": 0.25,
        # "built_area": 110468,
        "land_cost": 1500,
        "cost_build": 48_000,
        "price_sale": 92_000,
        "construction_years": 2,
        "sale_years": 3,
        "opex_rate": 800,
    },
    "residential_lowrise": {
        "density": 0.5,
        # "built_area": 294685,
        "land_cost": 1800,
        "cost_build": 50_000,
        "price_sale": 95_000,
        "construction_years": 2,
        "sale_years": 3,
        "opex_rate": 900,
    },
    "residential_midrise": {
        "density": 1.5,
        # "built_area": 277019,
        "land_cost": 2500,
        "cost_build": 55_000,
        "price_sale": 105_000,
        "construction_years": 3,
        "sale_years": 4,
        "opex_rate": 1200,
    },
    "residential_multistorey": {
        "density": 3.0,
        # "built_area": 686840,
        "land_cost": 3000,
        "cost_build": 62_000,
        "price_sale": 120_000,
        "construction_years": 4,
        "sale_years": 5,
        "opex_rate": 1500,
    },
    "business": {
        "density": 2.0,
        # "built_area": 1375709,
        "land_cost": 2800,
        "cost_build": 55_000,
        "rent_annual": 14_000,
        "rent_years": 12,
        "occupancy": 0.85,
        "construction_years": 3,
        "opex_rate": 1300,
    },
    "recreation": {
        "density": 0.2,
        # "built_area": 218180,
        "land_cost": 1000,
        "cost_build": 20_000,
        "rent_annual": 6_500,
        "rent_years": 15,
        "occupancy": 0.7,
        "construction_years": 2,
        "opex_rate": 1000,
    },
    "special": {
        "density": 1.0,
        # "built_area": 1164399,
        "land_cost": 1200,
        "cost_build": 35_000,
        "rent_annual": 8_000,
        "rent_years": 15,
        "occupancy": 0.8,
        "construction_years": 3,
        "opex_rate": 1500,
    },
    "industrial": {
        "density": 1.0,
        # "built_area": 409028,
        "land_cost": 900,
        "cost_build": 38_000,
        "rent_annual": 9_800,
        "rent_years": 12,
        "occupancy": 0.9,
        "construction_years": 2,
        "opex_rate": 700,
    },
    "agriculture": {
        "density": 0.1,
        # "built_area": 40104,
        "land_cost": 300,
        "cost_build": 25_000,
        "rent_annual": 2_500,
        "rent_years": 15,
        "occupancy": 0.95,
        "construction_years": 1,
        "opex_rate": 300,
    },
    "transport": {
        "density": 1.2,
        # "built_area": 709760,
        "land_cost": 1100,
        "cost_build": 18_000,
        "rent_annual": 4_200,
        "rent_years": 15,
        "occupancy": 0.88,
        "construction_years": 3,
        "opex_rate": 600,
    },
}



In [26]:
# project_gdf
# base_gdf
# score_gdf

In [27]:
project_gdf

,geometry,ip_type,ip_value,area
4,"POLYGON ((389429.389 6643590.914, 389460.139 6...",residential_midrise,1.2,228946.535860
2,"POLYGON ((388703.113 6643817.106, 388934.446 6...",residential_individual,0.7,555093.622474
6,"POLYGON ((388934.446 6643708.859, 389384.208 6...",industrial,1.5,338108.036466
1,"POLYGON ((388634.582 6644166.028, 388703.113 6...",residential_multistorey,0.4,560271.440910
3,"POLYGON ((388703.113 6643817.106, 388634.582 6...",transport,1.4,850917.184585
0,"POLYGON ((388634.582 6644166.028, 387644.932 6...",recreation,1.4,811927.195847
9,"POLYGON ((389460.139 6643570.686, 390172.418 6...",residential,1.5,441870.693343
8,"POLYGON ((390172.418 6643549.388, 390334.954 6...",special,1.5,409027.820815
5,"POLYGON ((389429.389 6643590.914, 389384.208 6...",business,0.8,817246.640517
10,"POLYGON ((390334.954 6643620.983, 390237.235 6...",agriculture,1.0,591467.024316


In [28]:
base_gdf = base_gdf.reset_index(drop=True)
base_gdf['id'] = base_gdf.index.astype('int64')
row_id = base_gdf.loc[base_gdf['id'] == 4]
row_id

,ip_type,ip_value,geometry,id
4,residential_multistorey,1.0,"POLYGON ((387861.999 6644712.938, 388202.551 6...",4


In [30]:
from urbanomy.methods.investment_potential import InvestmentAttractivenessAnalyzer

an = InvestmentAttractivenessAnalyzer(benchmarks=benchmarks_demo)
gdf_out, summary = an.calculate_investment_metrics(project_gdf)
summary

,NPV,IRR,ROI,PP_years,EI,investment_attractiveness,INV
profile,,,,,,,
residential_midrise,3321112192.45,0.18,3.18,6.24,66.06,1.20,35.53
residential_individual,1751356791.75,0.22,2.19,4.34,78.67,0.70,47.20
industrial,2720054613.57,0.16,3.94,10.32,61.28,1.50,44.21
residential_multistorey,6727134192.43,0.14,4.00,8.56,55.08,0.40,27.54
transport,-252206486.94,0.12,4.98,NaN,3.72,1.40,5.92
recreation,-380073423.50,0.10,2.88,NaN,1.09,1.40,6.44
residential,1549848933.82,0.24,2.28,4.24,83.37,1.50,55.02
special,-2445572906.57,0.09,3.90,NaN,0.00,1.50,3.75
business,2658405843.34,0.13,4.59,14.25,51.57,0.80,36.10
